# Baby Step 1 — Tax Planning Exo-Brain
## First governed tax-structure recommendation cycle

This notebook preserves Step 0, creates a Step 1 copy, and produces one synthetic Recommendation V1 for each of ten conglomerates. It is an educational architecture test—not tax or legal advice.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Governed design logic

The scoring model balances tax burden, withholding exposure, treaty reach, operational substance, documentation quality, and anti-avoidance stability. A low rate never overrides substance or the human gate.


In [ ]:
import csv
import hashlib
import json
import math
import re
import shutil
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path


STEP1_QUARTER = "2026-Q3"
STEP1_VERSION = "2026-Q3-R001"
STEP1_DATE = date(2026, 7, 21).isoformat()


def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows supplied for {path}")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


def fnum(value: str | int | float) -> float:
    return float(value)


def sensitivity_score(value: str) -> float:
    return {"low": 90.0, "medium": 65.0, "high": 35.0}[value]


def pattern_for(industry: str) -> str:
    industry_l = industry.lower()
    if any(k in industry_l for k in ("digital", "software", "media", "life sciences")):
        return "Substance-aligned IP, R&D, and service hub"
    if "financial" in industry_l:
        return "Regulated regional services and treasury alignment"
    if "consumer" in industry_l:
        return "Regional distribution and procurement alignment"
    return "Regional operating-principal and treasury alignment"


def apply_step1(source_vault: Path, output_vault: Path) -> dict:
    """Copy an immutable Step 0 baseline and add the first governed recommendation cycle."""
    source_vault = Path(source_vault)
    output_vault = Path(output_vault)
    required = [
        source_vault / "16_Data" / "tax_codes.csv",
        source_vault / "16_Data" / "jurisdictions.csv",
        source_vault / "16_Data" / "conglomerates.csv",
        source_vault / "16_Data" / "entities.csv",
        source_vault / "16_Data" / "intercompany_transactions.csv",
        source_vault / "13_Audit" / "STEP_0_SUCCESS.md",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Step 0 baseline is incomplete:\n" + "\n".join(missing))

    if output_vault.exists():
        shutil.rmtree(output_vault)
    shutil.copytree(source_vault, output_vault)

    data = output_vault / "16_Data"
    jurisdictions = read_csv(data / "jurisdictions.csv")
    tax_codes = read_csv(data / "tax_codes.csv")
    groups = read_csv(data / "conglomerates.csv")
    entities = read_csv(data / "entities.csv")
    transactions = read_csv(data / "intercompany_transactions.csv")

    if len(jurisdictions) != 20 or len(tax_codes) != 100 or len(groups) != 10:
        raise AssertionError("Step 1 requires the validated 20/100/10 Step 0 baseline.")

    jur_by_id = {row["jurisdiction_id"]: row for row in jurisdictions}
    code_by_key = {(row["jurisdiction_id"], row["module"]): row for row in tax_codes}
    entities_by_group: dict[str, list[dict]] = defaultdict(list)
    tx_by_group: dict[str, list[dict]] = defaultdict(list)
    for row in entities:
        entities_by_group[row["conglomerate_id"]].append(row)
    for row in transactions:
        tx_by_group[row["conglomerate_id"]].append(row)

    candidate_rows: list[dict] = []
    recommendation_rows: list[dict] = []
    rule_link_rows: list[dict] = []
    decision_rows: list[dict] = []

    for group in groups:
        gid = group["conglomerate_id"]
        gents = entities_by_group[gid]
        gtx = tx_by_group[gid]
        by_jur: dict[str, list[dict]] = defaultdict(list)
        for entity in gents:
            by_jur[entity["jurisdiction_id"]].append(entity)

        support_quality = Counter(row["support_status"] for row in gtx)
        documented_pct = 100.0 * support_quality["documented"] / max(1, len(gtx))
        documentation_score = min(100.0, 45.0 + 0.55 * documented_pct)
        candidates: list[dict] = []

        for jid, jents in sorted(by_jur.items()):
            jur = jur_by_id[jid]
            cit = code_by_key[(jid, "CIT")]
            wht = code_by_key[(jid, "WHT")]
            cfc = code_by_key[(jid, "CFC")]
            gmt = code_by_key[(jid, "GMT")]
            high = sum(e["substance_indicator"] == "high" for e in jents)
            medium = sum(e["substance_indicator"] == "medium" for e in jents)
            employees = sum(int(e["employees"]) for e in jents)
            key_functions = sum(
                e["role"] in {
                    "Regional holding", "IP owner", "R&D center", "Treasury company",
                    "Financing vehicle", "Distribution principal", "Manufacturing principal",
                }
                for e in jents
            )
            substance = min(100.0, high * 16.0 + medium * 8.0 + min(40.0, employees / 75.0) + key_functions * 5.0)
            tax_score = max(0.0, 100.0 - fnum(jur["headline_cit_rate_pct"]) / 35.0 * 100.0)
            wht_score = max(0.0, 100.0 - fnum(wht["numeric_parameter"]) / 35.0 * 100.0)
            treaty_score = min(100.0, fnum(jur["treaty_partner_count"]) / 10.0 * 100.0)
            anti_score = (sensitivity_score(cfc["change_sensitivity"]) + sensitivity_score(gmt["change_sensitivity"])) / 2.0
            total = (
                0.20 * tax_score + 0.15 * wht_score + 0.15 * treaty_score
                + 0.30 * substance + 0.10 * documentation_score + 0.10 * anti_score
            )
            eligible = substance >= 35.0
            candidates.append({
                "conglomerate_id": gid,
                "jurisdiction_id": jid,
                "jurisdiction": jur["name"],
                "headline_cit_rate_pct": jur["headline_cit_rate_pct"],
                "synthetic_wht_parameter": wht["numeric_parameter"],
                "treaty_partner_count": jur["treaty_partner_count"],
                "substance_score": round(substance, 2),
                "documentation_score": round(documentation_score, 2),
                "anti_avoidance_stability_score": round(anti_score, 2),
                "composite_score": round(total, 2),
                "eligible": str(eligible).upper(),
                "eligibility_reason": "Existing operational substance meets Step 1 threshold" if eligible else "Insufficient existing substance; candidate excluded",
            })

        candidates.sort(key=lambda row: (-fnum(row["composite_score"]), row["jurisdiction_id"]))
        eligible = [row for row in candidates if row["eligible"] == "TRUE"]
        selected = eligible[0] if eligible else candidates[0]
        selected["rank"] = 1
        for rank, candidate in enumerate(candidates, 1):
            candidate["rank"] = rank
            candidate["selected"] = str(candidate is selected).upper()
            candidate_rows.append(candidate)

        low_tax = min(candidates, key=lambda row: (fnum(row["headline_cit_rate_pct"]), row["jurisdiction_id"]))
        margin = fnum(candidates[0]["composite_score"]) - (fnum(candidates[1]["composite_score"]) if len(candidates) > 1 else 0.0)
        confidence = "HIGH" if fnum(selected["composite_score"]) >= 70 and margin >= 5 else "MEDIUM"
        rec_id = f"REC-{gid.split('-')[1]}-V001"
        decision_id = f"DEC-{gid.split('-')[1]}-R001"
        selected_jid = selected["jurisdiction_id"]
        selected_name = selected["jurisdiction"]
        home_jid = group["home_jurisdiction_id"]
        pattern = pattern_for(group["industry"])

        selected_roles = sorted({e["role"] for e in by_jur[selected_jid]})
        has_ip = any(role in selected_roles for role in ("IP owner", "R&D center"))
        has_treasury = any(role in selected_roles for role in ("Treasury company", "Financing vehicle"))
        partial_count = support_quality["partial"] + support_quality["to_be_tested"]
        low_tax_conflict = low_tax["jurisdiction_id"] != selected_jid
        counterfactual = (
            f"The lowest-rate footprint jurisdiction, {low_tax['jurisdiction']} ({low_tax['headline_cit_rate_pct']}%), was not selected because its governed composite score was {low_tax['composite_score']} versus {selected['composite_score']} for {selected_name}; rate alone cannot authorize a structure."
            if low_tax_conflict else
            f"{selected_name} is also the lowest-rate footprint jurisdiction, but selection remains conditional on its substance, documentation, anti-avoidance, and human-review constraints."
        )

        supporting_keys = [(selected_jid, "CIT"), (selected_jid, "WHT"), (selected_jid, "TP")]
        limiting_keys = [(selected_jid, "CFC"), (selected_jid, "GMT"), (home_jid, "CFC"), (home_jid, "GMT")]
        for relation, keys in (("SUPPORTING", supporting_keys), ("LIMITING", limiting_keys)):
            for sequence, key in enumerate(keys, 1):
                code = code_by_key[key]
                rule_link_rows.append({
                    "recommendation_id": rec_id,
                    "conglomerate_id": gid,
                    "relation": relation,
                    "sequence": sequence,
                    "tax_code_id": code["tax_code_id"],
                    "jurisdiction_id": code["jurisdiction_id"],
                    "module": code["module"],
                    "version": code["version"],
                    "effect": (
                        "Candidate design input; does not independently authorize implementation"
                        if relation == "SUPPORTING" else
                        "Countervailing constraint; unresolved exposure narrows permission"
                    ),
                })

        recommendation_rows.append({
            "recommendation_id": rec_id,
            "conglomerate_id": gid,
            "conglomerate": group["name"],
            "quarter": STEP1_QUARTER,
            "version": STEP1_VERSION,
            "status": "DRAFT_FOR_HUMAN_REVIEW",
            "design_pattern": pattern,
            "selected_hub_jurisdiction_id": selected_jid,
            "selected_hub_jurisdiction": selected_name,
            "composite_score": selected["composite_score"],
            "confidence": confidence,
            "decision_id": decision_id,
            "permission": "INTERNAL_SCENARIO_ONLY",
            "synthetic": True,
        })
        decision_rows.append({
            "decision_id": decision_id,
            "recommendation_id": rec_id,
            "conglomerate_id": gid,
            "status": "PENDING_HUMAN_REVIEW",
            "permitted_action": "REVIEW_AND_REQUEST_EVIDENCE_ONLY",
            "implementation_authorized": False,
            "synthetic": True,
        })

        candidate_table = "\n".join(
            f"| {c['rank']} | {c['jurisdiction']} | {c['headline_cit_rate_pct']}% | {c['substance_score']} | {c['composite_score']} | {c['eligible']} |"
            for c in candidates
        )
        supporting_lines = "\n".join(
            f"- [[{code_by_key[key]['tax_code_id']}_{key[1]}]] — {key[1]} design input, version {code_by_key[key]['version']}."
            for key in supporting_keys
        )
        limiting_lines = "\n".join(
            f"- [[{code_by_key[key]['tax_code_id']}_{key[1]}]] — {key[1]} countervailing constraint, version {code_by_key[key]['version']}."
            for key in limiting_keys
        )
        ip_action = (
            "Retain and test the existing IP/R&D functions in the hub; no migration is authorized."
            if has_ip else
            "Do not migrate IP: the selected hub lacks a demonstrated IP/R&D functional base in Step 1."
        )
        treasury_action = (
            "Retain and test existing treasury/financing functions in the hub, subject to capital and control evidence."
            if has_treasury else
            "Do not centralize treasury: the selected hub lacks a demonstrated financing function in Step 1."
        )
        rec_md = f"""
---
object_type: recommendation
recommendation_id: {rec_id}
conglomerate_id: {gid}
quarter: {STEP1_QUARTER}
version: {STEP1_VERSION}
status: DRAFT_FOR_HUMAN_REVIEW
permission: INTERNAL_SCENARIO_ONLY
confidence: {confidence}
synthetic: true
---

# {rec_id} — {group['name']}

> Synthetic analytical exercise only. This draft is not tax or legal advice and authorizes no transaction, filing, communication, restructuring, or implementation.

## Proposed design

**Pattern:** {pattern}  
**Candidate coordination hub:** {selected_name} ([[{selected_jid}]])  
**Governed composite score:** {selected['composite_score']} / 100  
**Status:** Draft for human review

1. Preserve the ultimate-parent location and all existing legal ownership until separately reviewed.
2. Use the existing {selected_name} footprint only as a scenario for regional coordination; no new entity is proposed at this step.
3. {ip_action}
4. {treasury_action}
5. Keep operating profits with the entities that perform functions, employ people, control risks, and own relevant assets.
6. Remediate the {partial_count} intercompany transactions whose support is partial or still to be tested before any design can advance.

## Candidate comparison

| Rank | Jurisdiction | Headline CIT | Substance | Composite | Eligible |
|---:|---|---:|---:|---:|---|
{candidate_table}

## Supporting rule set

{supporting_lines}

## Contrary and limiting rule set

{limiting_lines}

{counterfactual}

## Required evidence before qualification

- Functional interviews and decision-rights map for the proposed hub.
- Payroll, premises, board-control, capital, and risk-control evidence.
- Transaction-level transfer-pricing support and tested-party selection.
- Treaty entitlement and beneficial-ownership analysis.
- Home-jurisdiction CFC and global-minimum-tax calculations.
- Local counsel review using real, current law if the synthetic design is ever translated into practice.

## Decision gate

See [[{decision_id}]]. The only permitted action is internal review and evidence collection. Human approval cannot convert this synthetic record into real advice.
"""
        write_text(output_vault / "09_Recommendations" / f"{rec_id}.md", rec_md)

        decision_md = f"""
---
object_type: decision_gate
decision_id: {decision_id}
recommendation_id: {rec_id}
conglomerate_id: {gid}
status: PENDING_HUMAN_REVIEW
implementation_authorized: false
synthetic: true
---

# {decision_id} — Human Review Gate

## Recommendation under review

[[{rec_id}]] for {group['name']}.

## Available internal decisions

- **QUALIFY FOR STEP 2:** allow deeper synthetic testing of objectives, constraints, sensitivity, and fragility.
- **REQUEST EVIDENCE:** retain the draft but require specified synthetic evidence.
- **REOPEN:** reject the selected candidate and recompute alternatives.

## Prohibited decision

No reviewer may authorize a real transaction, filing position, communication, restructuring, or implementation from this synthetic exercise.
"""
        write_text(output_vault / "10_Decisions" / f"{decision_id}.md", decision_md)

        group_path_candidates = list((output_vault / "03_Conglomerates").glob(f"{gid}_*.md"))
        if len(group_path_candidates) != 1:
            raise AssertionError(f"Expected one conglomerate note for {gid}")
        group_path = group_path_candidates[0]
        group_text = group_path.read_text(encoding="utf-8")
        group_text = re.sub(r"recommendation_status: NOT_DEVELOPED", "recommendation_status: DRAFT_FOR_HUMAN_REVIEW", group_text, count=1)
        group_text = re.sub(r"recommendation_version: NONE", f"recommendation_version: {STEP1_VERSION}", group_text, count=1)
        group_text += f"\n## Step 1 recommendation\n\n- [[{rec_id}]]\n- [[{decision_id}]]\n"
        write_text(group_path, group_text)

    for group in groups:
        group["recommendation_status"] = "DRAFT_FOR_HUMAN_REVIEW"
        group["recommendation_version"] = STEP1_VERSION

    write_csv(data / "conglomerates.csv", groups)
    write_csv(data / "recommendations.csv", recommendation_rows)
    write_csv(data / "recommendation_candidate_scores.csv", candidate_rows)
    write_csv(data / "recommendation_rule_links.csv", rule_link_rows)
    write_csv(data / "decision_gates.csv", decision_rows)

    register_lines = "\n".join(
        f"| [[{row['recommendation_id']}]] | {row['conglomerate']} | {row['design_pattern']} | {row['selected_hub_jurisdiction']} | {row['composite_score']} | {row['confidence']} | [[{row['decision_id']}]] |"
        for row in recommendation_rows
    )
    write_text(output_vault / "12_Reports" / "STEP_1_RECOMMENDATION_REGISTER.md", f"""
---
object_type: portfolio_report
report_id: RPT-STEP-1-001
quarter: {STEP1_QUARTER}
status: INTERNAL_SYNTHETIC_DRAFT
synthetic: true
---

# Step 1 Recommendation Register

The first governed structure-design loop produced one Recommendation V1 for each of the ten synthetic conglomerates. The register is an internal analytical baseline, not a tax plan for implementation.

| Recommendation | Conglomerate | Design pattern | Candidate hub | Score | Confidence | Gate |
|---|---|---|---|---:|---|---|
{register_lines}

## Portfolio controls

- All ten recommendations are pending human review.
- Exactly seven versioned rule links support or constrain each recommendation.
- Rate minimization never overrides substance, anti-avoidance, documentation, or governance.
- Step 0 remains unchanged; Step 1 is a copied and versioned state.
- No tax-code changes or new companies occur until the later quarterly update step.
""")

    write_text(output_vault / "11_Quarterly_Updates" / STEP1_QUARTER / "STEP_1_FIRST_CYCLE.md", f"""
# {STEP1_QUARTER} — First Recommendation Cycle

- Baseline inherited from Step 0: 100 unchanged tax-code modules, 20 fictional jurisdictions, and 10 conglomerates.
- New analytical output: 10 draft Recommendation V1 records.
- New experience base: candidate rankings, supporting rules, limiting rules, constraints, and pending decision gates.
- Tax-code alterations this cycle: 0.
- New companies this cycle: 0.
- Implementation authority: none.
""")

    write_text(output_vault / "14_Hot_Cache" / "CURRENT_STATE.md", f"""
# Current State — {STEP1_QUARTER} Recommendation V1

- Active step: 1 of 10
- Tax-code modules: 100 (unchanged)
- Fictional jurisdictions: 20 (unchanged)
- Conglomerates: 10 (unchanged)
- Draft recommendations: 10
- Current recommendation version: {STEP1_VERSION}
- Decision gates pending: 10
- Formal provenance/atomic-claim layer: scheduled for Step 3
- Next permitted experiment: Step 2, business-model generalization and fragility testing
- Explicit prohibition: no real tax advice, external action, filing, communication, or restructuring
""")

    state = {
        "project": "Tax Planning Exo-Brain",
        "active_step": 1,
        "quarter": STEP1_QUARTER,
        "recommendation_version": STEP1_VERSION,
        "counts": {
            "jurisdictions": len(jurisdictions),
            "tax_codes": len(tax_codes),
            "conglomerates": len(groups),
            "entities": len(entities),
            "transactions": len(transactions),
            "recommendations": len(recommendation_rows),
            "candidate_scores": len(candidate_rows),
            "rule_links": len(rule_link_rows),
            "decision_gates": len(decision_rows),
        },
        "permission": "INTERNAL_SCENARIO_ONLY",
        "implementation_authorized": False,
        "synthetic": True,
    }
    write_text(output_vault / "00_System" / "CURRENT_STATE.json", json.dumps(state, indent=2))

    manifest_rows = []
    excluded = {"13_Audit/STEP_1_VALIDATION.json", "13_Audit/STEP_1_MANIFEST.csv", "13_Audit/STEP_1_SUCCESS.md"}
    for path in sorted(p for p in output_vault.rglob("*") if p.is_file()):
        rel = path.relative_to(output_vault).as_posix()
        if rel in excluded:
            continue
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        manifest_rows.append({"path": rel, "bytes": path.stat().st_size, "sha256": digest})
    write_csv(output_vault / "13_Audit" / "STEP_1_MANIFEST.csv", manifest_rows)

    checks = {
        "step0_success_marker_present": (output_vault / "13_Audit" / "STEP_0_SUCCESS.md").exists(),
        "jurisdictions_equal_20": len(jurisdictions) == 20,
        "tax_codes_equal_100": len(tax_codes) == 100,
        "conglomerates_equal_10": len(groups) == 10,
        "recommendations_equal_10": len(recommendation_rows) == 10,
        "one_recommendation_per_group": len({r["conglomerate_id"] for r in recommendation_rows}) == 10,
        "all_recommendations_draft": all(r["status"] == "DRAFT_FOR_HUMAN_REVIEW" for r in recommendation_rows),
        "seven_rule_links_per_recommendation": all(v == 7 for v in Counter(r["recommendation_id"] for r in rule_link_rows).values()),
        "decision_gates_equal_10": len(decision_rows) == 10,
        "all_decisions_pending": all(r["status"] == "PENDING_HUMAN_REVIEW" for r in decision_rows),
        "implementation_never_authorized": not any(str(r["implementation_authorized"]).lower() == "true" for r in decision_rows),
        "no_tax_code_versions_changed": all(r["version"] == "2026-Q3-V001" for r in tax_codes),
        "no_new_companies": len(groups) == 10,
        "formal_claim_layer_still_empty": not any((output_vault / "07_Atomic_Claims").glob("*.md")),
        "formal_contradiction_layer_still_empty": not any((output_vault / "08_Contradictions").glob("*.md")),
    }
    validation = {
        "step": 1,
        "date": STEP1_DATE,
        "quarter": STEP1_QUARTER,
        "recommendation_version": STEP1_VERSION,
        "checks": checks,
        "counts": state["counts"],
        "status": "PASS" if all(checks.values()) else "FAIL",
    }
    write_text(output_vault / "13_Audit" / "STEP_1_VALIDATION.json", json.dumps(validation, indent=2))
    if validation["status"] != "PASS":
        failed = [key for key, ok in checks.items() if not ok]
        raise AssertionError("Step 1 validation failed: " + ", ".join(failed))
    write_text(output_vault / "13_Audit" / "STEP_1_SUCCESS.md", f"""
# Step 1 Validation: PASS

Validated {len(recommendation_rows)} governed Recommendation V1 records, {len(rule_link_rows)} rule links, and {len(decision_rows)} pending human decision gates. No implementation is authorized.
""")
    return validation


In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project')
SOURCE_VAULT = PROJECT / 'Step_0' / 'Tax_Planning_ExoBrain_Vault'
OUTPUT_VAULT = PROJECT / 'Step_1' / 'Tax_Planning_ExoBrain_Vault'
validation = apply_step1(SOURCE_VAULT, OUTPUT_VAULT)
print(json.dumps(validation, indent=2))
print(f'\nStep 1 vault created at: {OUTPUT_VAULT}')


## Expected result

The validation status must be `PASS`: 20 jurisdictions, 100 unchanged tax-code modules, 10 conglomerates, 10 draft recommendations, 70 rule links, and 10 pending human decision gates. No implementation is authorized.
